# Fraud Detection Agentic Knowledge Graph Demo

This notebook is designed for a beginner classroom demo. Students will see how a small fraud-detection knowledge graph is defined, validated, loaded into KuzuDB, queried with Cypher, visualized, and used by a simple graph agent.

The business story is simple: a fraud operations team wants to understand which detection methods use which indicators, which data sources they analyze, and which fraud types they detect with high confidence.

## 0. One-time setup

Run these commands from the repository root before opening the notebook:

```bash
cd python-agentic-knowledge-graph
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
python -m pip install --upgrade pip
pip install -r requirements-demo.txt
python -m jupyter lab notebooks/fraud_knowledge_graph_demo.ipynb
```

If you are already inside Jupyter, start at the next cell.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

OUTPUT_ROOT = PROJECT_ROOT / 'output'
FRAUD_DB = OUTPUT_ROOT / 'fraud.kuzu'

PROJECT_ROOT

In [ ]:
from IPython.display import HTML, Markdown, SVG, display

from agentic_knowledge_graph.agent import GraphAgent
from agentic_knowledge_graph.db import connect, execute
from agentic_knowledge_graph.domains import get_domain
from agentic_knowledge_graph.loader import build_domain, load_dataset
from agentic_knowledge_graph.validation import validate_domain
from agentic_knowledge_graph.visualization import (
    fraud_graph_svg,
    records_to_html,
    table_to_html,
    write_fraud_graph_html,
)

spec = get_domain('fraud')
dataset = load_dataset('fraud')
spec.description

## 1. What are we building?

A knowledge graph stores entities as **nodes** and business connections as **relationships**. In this fraud example:

- `DetectionMethod` nodes represent methods such as Machine Learning, Rule Engine, and Network Analysis.
- `Indicator` nodes represent signals such as Velocity Spike or New Device.
- `DataSource` nodes represent systems such as Transaction Data or Device Data.
- `FraudType` nodes represent outcomes such as Account Takeover or Payment Fraud.

The useful part is not just the list of values. The graph lets us ask connected questions: which method uses which indicator, which source supports it, and which fraud type it can detect.

In [ ]:
node_schema = [
    {
        'Node table': node.name,
        'Primary key': node.primary_key,
        'Columns': ', '.join(f'{name}: {kind}' for name, kind in node.columns),
    }
    for node in spec.nodes
]

relationship_schema = [
    {
        'Relationship': rel.name,
        'From': rel.from_node,
        'To': rel.to_node,
        'Properties': ', '.join(f'{name}: {kind}' for name, kind in rel.columns) or '(none)',
    }
    for rel in spec.relationships
]

display(Markdown('### Node tables'))
display(HTML(records_to_html(node_schema, ('Node table', 'Primary key', 'Columns'))))
display(Markdown('### Relationship tables'))
display(HTML(records_to_html(relationship_schema, ('Relationship', 'From', 'To', 'Properties'))))

## 2. Read the schema as sentences

Ask students to translate each relationship into a sentence:

- `DetectionMethod -[Detects]-> FraudType`: Network Analysis detects Transaction Fraud with confidence 0.92.
- `DetectionMethod -[Uses]-> Indicator`: Device Intelligence uses New Device with weight 0.95.
- `DetectionMethod -[Analyzes]-> DataSource`: Machine Learning analyzes Transaction Data with priority 1.

This sentence test helps beginners understand relationship direction. The relationship property belongs to the connection, not to either node by itself.

In [ ]:
summary_rows = []
for label, rows in dataset['nodes'].items():
    summary_rows.append({'Section': 'Node', 'Name': label, 'Rows': len(rows)})
for label, rows in dataset['relationships'].items():
    summary_rows.append({'Section': 'Relationship', 'Name': label, 'Rows': len(rows)})

issues = validate_domain(spec)

display(Markdown('### Dataset size'))
display(HTML(records_to_html(summary_rows, ('Section', 'Name', 'Rows'))))

if issues:
    display(Markdown('### Validation issues'))
    display(HTML(records_to_html([{'Domain': i.domain, 'Issue': i.message} for i in issues], ('Domain', 'Issue'))))
else:
    display(Markdown('**Validation passed.** Every relationship points to an existing source and target node, and required properties are present.'))

## 3. Visualize before building

This diagram is created directly from `data/fraud.json`. It is useful before database loading because students can compare the JSON structure with the graph shape.

In [ ]:
display(SVG(fraud_graph_svg()))

## 4. Build the Kuzu database

`build_domain('fraud', FRAUD_DB, reset=True)` performs the same work as the CLI command:

```bash
akg build fraud --reset
```

Behind the scenes it creates node tables, creates relationship tables, inserts nodes, then inserts relationships by matching `from` and `to` primary-key values.

In [ ]:
nodes_loaded, relationships_loaded = build_domain('fraud', FRAUD_DB, reset=True)
_database, connection = connect(FRAUD_DB)

display(Markdown(f'Built `{FRAUD_DB}` with **{nodes_loaded} nodes** and **{relationships_loaded} relationships**.'))

In [ ]:
def show_query(number: int):
    query = spec.queries[number - 1]
    display(Markdown(f'### Query {number}: {query.title}'))
    display(Markdown(f'Teaching point: **{query.teaching_point}**'))
    display(Markdown(f'```cypher\n{query.cypher}\n```'))
    table = execute(connection, query.cypher)
    display(HTML(table_to_html(table)))
    return table

## 5. Run beginner-friendly graph questions

Start with one-hop traversal. The graph pattern means: find a detection method, follow the `Detects` relationship, and reach a fraud type.

In [ ]:
methods_to_fraud = show_query(1)

Now add a confidence threshold. This is closer to an operational question: which detections are strong enough to show to an analyst first?

In [ ]:
high_confidence = show_query(5)

Next, show a complete pipeline. This query connects four entities: data source, method, indicator, and fraud type.

In [ ]:
complete_pipeline = show_query(8)

## 6. Create a meaningful outcome

For a demo, avoid stopping at raw rows. Turn graph evidence into a decision:

**Which fraud type should the team prioritize for analyst review, based on highest detection confidence and method coverage?**

In [ ]:
outcome_cypher = '''
MATCH (m:DetectionMethod)-[r:Detects]->(f:FraudType)
WITH f, COUNT(m) AS detecting_methods, MAX(r.confidence) AS best_confidence
RETURN f.name, f.severity, detecting_methods, best_confidence
ORDER BY best_confidence DESC, detecting_methods DESC
'''

outcome = execute(connection, outcome_cypher)
display(Markdown('### Prioritized fraud-review candidates'))
display(HTML(table_to_html(outcome)))
display(Markdown(
    'Teaching note: the highest row is not just a value. It becomes an operational recommendation: '
    'start the analyst review queue with the fraud type that has the strongest graph-backed detection evidence.'
))

## 7. Ask the graph agent

The agent in this repository is deliberately simple and explainable:

1. Observe the question.
2. Choose the domain, or use the domain supplied by the caller.
3. Select a retrieval query.
4. Run Cypher against Kuzu.
5. Answer only from the graph evidence.

This is the core GraphRAG teaching idea: retrieval comes from connected graph evidence before answer generation.

In [ ]:
agent = GraphAgent(database_root=OUTPUT_ROOT)

questions = [
    'Which fraud detection methods have high confidence?',
    'Which data sources are used for fraud source coverage?',
    'Which indicators are reused by multiple detection methods?',
]

for question in questions:
    response = agent.ask(question, domain='fraud')
    display(Markdown(f'### Question: {question}'))
    display(Markdown(response.answer))
    display(HTML(f'<details><summary>Show graph context</summary><pre>{response.context}</pre></details>'))

## 8. Optional mini chatbot cell

Use this cell during the live demo after students understand the fixed examples. Try questions such as:

- Which methods use indicators?
- Which fraud methods have high confidence?
- Which data sources support fraud detection?
- Which indicators are shared?

In [ ]:
try:
    import ipywidgets as widgets

    question_box = widgets.Textarea(
        value='Which fraud methods have high confidence?',
        description='Ask:',
        layout=widgets.Layout(width='100%', height='80px'),
    )
    ask_button = widgets.Button(description='Ask graph', button_style='primary')
    answer_area = widgets.Output()

    def ask_graph(_button):
        with answer_area:
            answer_area.clear_output()
            response = agent.ask(question_box.value, domain='fraud')
            display(Markdown(response.answer))
            display(HTML(f'<details><summary>Show retrieved graph context</summary><pre>{response.context}</pre></details>'))

    ask_button.on_click(ask_graph)
    display(question_box, ask_button, answer_area)
except ImportError:
    display(Markdown('Install the demo extras with `pip install -r requirements-demo.txt` to enable widgets.'))

## 9. Save a standalone visualization

This gives you a backup artifact for a classroom or presentation deck.

In [ ]:
graph_html_path = write_fraud_graph_html(OUTPUT_ROOT / 'fraud_knowledge_graph.html')
display(Markdown(f'Saved standalone visualization to `{graph_html_path}`.'))

## 10. Student exercises

1. Add a new indicator called `Impossible Travel` to `data/fraud.json`.
2. Connect `Device Intelligence` to that indicator with a `Uses` relationship.
3. Run validation again and rebuild the graph.
4. Rerun the visualization and explain what changed.
5. Add a new Cypher query to `domains.py` that answers: which methods combine device and location indicators?

Discussion prompt: what extra production properties would you add? Good answers include source system, timestamp, model version, owner, approval status, and explanation text.